# Futures data quality: IB API vs DB rolling view

Wraps `scripts/futures_data_quality.py` -- six-step diagnostic:
1. Stale price detection (consecutive zero-return runs)
2. Volume/liquidity gate (if volume available)
3. Roll date contamination (auto-detected or provided)
4. Listwise vs pairwise deletion assessment
5. Covariance matrix comparison (IB pairwise, IB listwise, DB listwise)
6. Structured recommendation

Computation is polars-based throughout. Conversion to pandas happens only
at the `pypfopt.risk_models.sample_cov` call site in Step 5 (consistent
with project CLAUDE.md convention).

Requires a live IB connection for the fetch step.

In [15]:
import sys
import logging
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import hvplot.polars   # registers .hvplot on polars DataFrames/Series
import polars as pl

from scripts.futures_data_quality import (
    fetch_ib_prices, load_db_prices, run_quality_check
)

pl.Config.set_tbl_rows(1000)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(1000)


polars.config.Config

## Parameters — edit here

In [16]:
# Edit INSTRUMENTS to narrow the universe -- delete symbols you don't want.
# INSTRUMENTS = 'MES,MNQ,MCL,MZL,MZC,MZS,MZW,MGC,SIL,J7,BRE,6M'
INSTRUMENTS      = 'ES,NQ'
DURATION         = '3 y'    # IB historical window
INCLUDE_DB       = False    # set True to also load from the local duckdb cache
STALE_THRESHOLD  = 3        # consecutive zero-return days to flag
OUTLIER_SIGMA    = 5.0      # σ for auto roll-date detection
HALFLIFE         = 60.0     # EWM covariance halflife (days)
ROLL_DATES       = None     # dict {ticker: [date_strings]} or None

HOST      = '127.0.0.1'
PORT      = 7496
CLIENT_ID = 22

In [17]:
# Fetch IB data.
from ib_tools.ibpysync import IBPySync
from options_bt.live.run_tsmom_rebalance import KNOWN_INSTRUMENTS, _build_instruments

instruments_spec = globals().get('INSTRUMENTS', ','.join(sorted(KNOWN_INSTRUMENTS)))
instruments = _build_instruments(instruments_spec, None, 15)

print(f'instruments_spec : {instruments_spec!r}')
print(f'instruments ({len(instruments)}):')
for instr in instruments:
    sym      = instr.get('symbol')
    ib_sym   = instr.get('ib_symbol', sym)
    sig_sym  = instr.get('signal_symbol', ib_sym)
    exchange = instr.get('exchange', 'CME')
    print(f'  {sym:8s}  ib={ib_sym}  signal={sig_sym}  exchange={exchange}')

print(f'\nConnecting to IB at {HOST}:{PORT} (clientId={CLIENT_ID}) ...')
ib = IBPySync()
ib.connect(HOST, PORT, CLIENT_ID)
print('Connected.')
try:
    ib_prices, ib_volume = fetch_ib_prices(ib, instruments, duration=DURATION)
except Exception:
    import traceback
    traceback.print_exc()
    raise
finally:
    ib.disconnect()
    print('Disconnected.')

fetched = [c for c in ib_prices.columns if c != 'date']
print(f'\nFetched {len(fetched)}/{len(instruments)} instruments, {len(ib_prices)} rows')
print(f'Columns : {fetched}')
print(f'Volume  : {"yes" if ib_volume is not None else "not available"}')
ib_prices.head()

INFO ib_insync.client: Connecting to 127.0.0.1:7496 with clientId 22...


instruments_spec : 'ES,NQ'
instruments (2):
  ES        ib=ES  signal=ES  exchange=CME
  NQ        ib=NQ  signal=NQ  exchange=CME

Connecting to IB at 127.0.0.1:7496 (clientId=22) ...


INFO ib_insync.client: Connected
INFO ib_insync.client: Logged on to server version 176
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:cafarm
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:hfarm
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:eufarmnj
INFO ib_insync.client: API connection ready
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:cashfarm
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:usfuture
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:usfuture.nj
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:afarm
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:usopt.nj
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market data farm connection is OK:jfarm
INFO ib_insync.wrapper: Warning 2104, reqId -1: Market da

Connected.


INFO scripts.futures_data_quality: ES: fetching ES continuous bars...
INFO scripts.futures_data_quality: NQ: qualifying contract (NQ @ CME)...
INFO scripts.futures_data_quality: NQ: fetching NQ continuous bars...
INFO ib_insync.ib: Disconnecting from 127.0.0.1:7496, 396 B sent in 12 messages, 129 kB received in 392 messages, session time 8.69 s.
INFO ib_insync.client: Disconnecting
INFO ib_insync.client: Disconnected.


Disconnected.

Fetched 2/2 instruments, 748 rows
Columns : ['ES', 'NQ']
Volume  : yes


date,ES,NQ
date,f64,f64
2023-07-05,5036.0,17564.0
2023-07-06,4996.25,17430.25
2023-07-07,4982.25,17368.25
2023-07-10,4992.0,17353.0
2023-07-11,5024.25,17429.0


In [18]:
ib_volume

date,ES,NQ
date,f64,f64
2023-07-05,0.0,0.0
2023-07-06,0.0,0.0
2023-07-07,0.0,0.0
2023-07-10,0.0,0.0
2023-07-11,0.0,0.0
2023-07-12,0.0,0.0
2023-07-13,0.0,0.0
2023-07-14,0.0,0.0
2023-07-17,0.0,0.0


In [32]:
ib_volume.describe()

statistic,date,ES,NQ
str,str,f64,f64
"""count""","""748""",748.0,748.0
"""null_count""","""0""",0.0,0.0
"""mean""","""2025-01-01 04:42:59.679144""",827250.854278,302676.066845
"""std""",null,642121.965627,226561.688618
"""min""","""2023-07-05""",0.0,0.0
"""25%""","""2024-04-05""",416.0,309.0
"""50%""","""2025-01-02""",973076.0,374804.0
"""75%""","""2025-10-01""",1.262796e6,457910.0
"""max""","""2026-07-01""",2.787467e6,909998.0


In [36]:
from datetime import date

df_2y = ib_volume.filter(
    pl.col('date') > date(2024,1,1)
)
df_2y

date,ES,NQ
date,f64,f64
2024-01-02,2.0,0.0
2024-01-03,3.0,0.0
2024-01-08,3.0,2.0
2024-01-09,1.0,1.0
2024-01-11,19.0,2.0
2024-01-12,4.0,2.0
2024-01-16,23.0,0.0
2024-01-17,2.0,4.0
2024-01-18,114.0,3.0


In [37]:
df_2y.describe()

statistic,date,ES,NQ
str,str,f64,f64
"""count""","""623""",623.0,623.0
"""null_count""","""0""",0.0,0.0
"""mean""","""2025-04-03 03:00:17.335473""",993231.780096,363405.579454
"""std""",null,574508.680474,198848.783566
"""min""","""2024-01-02""",1.0,0.0
"""25%""","""2024-08-20""",812084.0,320482.0
"""50%""","""2025-04-03""",1.050119e6,401024.0
"""75%""","""2025-11-14""",1.325236e6,482400.0
"""max""","""2026-07-01""",2.787467e6,909998.0


In [38]:
ib_prices.describe()

statistic,date,ES,NQ
str,str,f64,f64
"""count""","""748""",748.0,748.0
"""null_count""","""0""",0.0,0.0
"""mean""","""2025-01-01 04:42:59.679144""",6150.930816,22321.527072
"""std""",null,746.576363,3383.859194
"""min""","""2023-07-05""",4599.5,16048.5
"""25%""","""2024-04-05""",5621.0,19970.5
"""50%""","""2025-01-02""",6178.25,22039.75
"""75%""","""2025-10-01""",6742.0,24883.0
"""max""","""2026-07-01""",7687.75,31015.75


In [39]:
ib_prices

date,ES,NQ
date,f64,f64
2023-07-05,5036.0,17564.0
2023-07-06,4996.25,17430.25
2023-07-07,4982.25,17368.25
2023-07-10,4992.0,17353.0
2023-07-11,5024.25,17429.0
2023-07-12,5062.0,17625.0
2023-07-13,5097.5,17916.5
2023-07-14,5094.25,17903.5
2023-07-17,5111.5,18062.25


In [19]:
# Optionally load DB prices for comparison.
db_prices = None
if INCLUDE_DB:
    tickers = [instr['symbol'] for instr in instruments]
    db_prices = load_db_prices(tickers)
    if db_prices is not None:
        print(f'DB prices loaded: {len(db_prices)} rows, columns: {db_prices.columns}')
    else:
        print('DB prices unavailable -- proceeding with IB only')

## Run quality check (Steps 1–6)

In [20]:
results = run_quality_check(
    ib_prices, db_prices,
    volume=ib_volume,
    roll_dates=ROLL_DATES,
    stale_run_threshold=STALE_THRESHOLD,
    outlier_sigma=OUTLIER_SIGMA,
    halflife=HALFLIFE,
    print_report=True,
)

STEP 1 — STALE PRICE DETECTION
  No stale runs detected.

STEP 2 — VOLUME / LIQUIDITY GATE
  ES: 235 low-volume rows (< 10% of median daily volume)
  NQ: 235 low-volume rows (< 10% of median daily volume)
  Zero-return + low-volume overlap: 0
  Zero-return at normal volume (holidays/CBs): 0

STEP 3 — ROLL DATE CONTAMINATION
  Auto-detected probable roll/anomaly dates: 5

STEP 4 — LISTWISE VS PAIRWISE DELETION
  Total rows: 748
  Listwise rows: 744 (99.5%)
  Pairwise N CV: 0.00% (OK)

STEP 5 — COVARIANCE MATRIX COMPARISON
  Pairwise vs listwise — pairs with >1% diff: 0

  Annualised vol comparison:
    ib_pairwise  ib_listwise  pw_vs_lw_diff
ES       0.1497       0.1493         0.0003
NQ       0.2015       0.2009         0.0006

DATA SOURCE RECOMMENDATION
--------------------------
Preferred source: IB
Reason: No DB source provided -- using IB data only.

INSTRUMENTS TO REVIEW
---------------------
ES: 3 probable roll/anomaly dates auto-detected — review and add to roll_dates if confirm

In [21]:
# Unpack the key artifacts.
stale   = results['stale']
rolls   = results['rolls']
deletion = results['deletion']
cov     = results['covariance']
S_final = results['S_final']    # recommended covariance matrix (pandas)
ret_final = results['ret_final']  # corresponding returns (pandas)

## Stale price runs

In [22]:
if stale['flagged_tickers']:
    print('Instruments with stale price runs:')
    for ticker, n_rows in sorted(stale['flagged_tickers'].items()):
        print(f'  {ticker}: {n_rows} flagged rows')
        for start, end, length in stale['runs'][ticker]:
            print(f'    {start} -> {end} ({length} consecutive zeros)')
else:
    print('No stale price runs detected.')

No stale price runs detected.


In [23]:
# Stale-price timeline -- all-polars: unpivot then .hvplot directly.
stale_mask = stale['stale_mask']
tickers = [c for c in stale_mask.columns if c != 'date']
if any(stale_mask[t].sum() > 0 for t in tickers):
    stale_long = (
        stale_mask
        .with_columns(
            [pl.col(t).cast(pl.Int8) for t in tickers]
            + [pl.col('date').cast(pl.Utf8)]
        )
        .unpivot(index='date', on=tickers, variable_name='instrument', value_name='stale')
    )
    stale_long.hvplot.heatmap(
        x='date', y='instrument', C='stale',
        cmap='Reds', clim=(0, 1), colorbar=False,
        width=900, height=max(200, len(tickers) * 40),
        title=f'Stale-price mask (red = zero-return run ≥ {STALE_THRESHOLD} days)',
        rot=45,
    )
else:
    print('No stale periods to plot.')

No stale periods to plot.


## Auto-detected roll / anomaly dates

In [24]:
total_auto = sum(len(v) for v in rolls['auto_detected'].values())
print(f'Auto-detected roll/anomaly dates: {total_auto} across {len(rolls["auto_detected"])} instruments')
for ticker, dates in rolls['auto_detected'].items():
    if dates:
        print(f'  {ticker}: {len(dates)} dates — {dates[:5]}{" ..." if len(dates) > 5 else ""}')

Auto-detected roll/anomaly dates: 5 across 2 instruments
  ES: 3 dates — [datetime.date(2025, 4, 4), datetime.date(2025, 4, 9), datetime.date(2025, 10, 10)]
  NQ: 2 dates — [datetime.date(2025, 4, 9), datetime.date(2025, 10, 10)]


## Correlation matrix (IB listwise)

In [25]:
if cov is not None:
    import numpy as np
    S = cov['S_ib_listwise']          # pandas from pypfopt -- convert at the plot boundary
    std = np.sqrt(np.diag(S.values))
    cm  = S / np.outer(std, std)
    cm_long = (
        pl.from_pandas(cm.reset_index().rename(columns={'index': 'instrument'}))
        .unpivot(index='instrument', variable_name='col', value_name='corr')
    )
    cm_long.hvplot.heatmap(
        x='col', y='instrument', C='corr',
        cmap='coolwarm', clim=(-1, 1), colorbar=True,
        width=500, height=450,
        title=f'Correlation matrix (IB listwise, EWM halflife={HALFLIFE}d)',
    )
else:
    print('Covariance comparison not available (pypfopt not installed).')

## Annualised vol comparison (pairwise vs listwise, IB vs DB)

In [26]:
if cov is not None:
    print(cov['vol_comparison'].round(4).to_string())
    if cov['flagged_vol_instruments']:
        print(f'\nInstruments with >5pp vol diff between pairwise and listwise: {cov["flagged_vol_instruments"]}')

    ib_pairwise  ib_listwise  pw_vs_lw_diff
ES       0.1497       0.1493         0.0003
NQ       0.2015       0.2009         0.0006


## Deletion assessment

In [27]:
print(f'Total rows:    {deletion["total_rows"]}')
print(f'Listwise rows: {deletion["listwise_rows"]} ({deletion["listwise_pct"]}%)')
print(f'Pairwise N CV: {deletion["pairwise_n_cv"]:.2%} '
      f'({"reliable" if deletion["pairwise_reliable"] else "UNRELIABLE"})')
print(f'Recommendation: {deletion["recommended_strategy"]}')

Total rows:    748
Listwise rows: 744 (99.5%)
Pairwise N CV: 0.00% (reliable)
Recommendation: listwise


In [28]:
# Per-pair N heatmap -- polars DataFrame built directly from the dict.
if deletion['pairwise_n']:
    rows = []
    for (ta, tb), n in deletion['pairwise_n'].items():
        rows += [{'x': ta, 'y': tb, 'N': float(n)},
                 {'x': tb, 'y': ta, 'N': float(n)}]
    tickers_all = sorted(set(t for pair in deletion['pairwise_n'] for t in pair))
    for t in tickers_all:
        rows.append({'x': t, 'y': t, 'N': float(deletion['total_rows'])})
    pl.DataFrame(rows).hvplot.heatmap(
        x='x', y='y', C='N',
        cmap='YlOrRd_r', colorbar=True,
        width=500, height=450,
        title='Per-pair N under pairwise deletion (lower = more data loss)',
    )